# 07 — Winger Playstyle Archetype Discovery

## Research Question

Do the attacking behaviours of wingers form meaningful,
distinguishable playstyle groups, or are they better represented
as continuous behavioural dimensions?

## Research Context

The previous notebooks established:

**Notebook 04 — Behavioural Representation**

- 1,396 player-club-season observations.
- 665 distinct footballers.
- Seven seasons across Europe's Big Five leagues.
- 20 standardized attacking-behaviour features.
- Eight behavioural macro-families.
- A family-balanced alternative representation.

**Notebook 05 — Playstyle Structure**

PCA identified five dominant behavioural dimensions
explaining 85.23% of total feature variance.

The resulting projections exhibited broadly continuous
distributions without visually obvious separated groups.

However, two-dimensional projections cannot establish
whether meaningful groups exist in the full feature space.

**Notebook 06 — Player Similarity**

A 20-feature Euclidean distance model identified stylistically
similar historical player spells.

The model demonstrated substantial neighbour-set agreement
under family-balanced Euclidean and standardized Manhattan
sensitivity alternatives.

The similarity analysis also identified unusually isolated
player profiles, particularly Jérémy Doku's 2023–24 season.

## Objective

Investigate whether clustering provides meaningful additional
information beyond continuous playstyle dimensions and
nearest-neighbour similarity.

Any proposed archetypes must demonstrate:

1. Distinguishable behavioural signatures.
2. Reasonable within-group consistency.
3. Interpretable relationships between groups.
4. Robustness under reasonable modelling alternatives.
5. Football interpretations supported by actual feature values.

Cluster assignments will not automatically be interpreted
as fixed or exclusive footballing identities.

## Analytical Principle

The existence and number of archetypes are empirical questions.

We will not assume that every winger belongs to a naturally
distinct attacking role.

If the data does not support robust discrete groupings,
a continuous playstyle framework remains an acceptable
and potentially more accurate research conclusion.


## 1. Modelling Boundaries

### Observational Unit

The observational unit remains a player-club-season spell.

A footballer may appear several times, reflecting different
seasons or club contexts.

These observations will initially remain separate because
attacking behaviour may evolve across seasons.

However, repeated observations of the same footballer are
not statistically independent.

We will therefore assess whether repeated-player representation
materially affects any eventual clustering results.

### Primary Representation

The primary modelling space contains all 20 globally
standardized attacking-behaviour features from Notebook 04.

We will not automatically reduce these features to the first
five principal components.

Notebook 05 showed that the five-component PCA representation
retained approximately 54.2% of the original top-ten neighbours,
despite explaining 85.23% of the total variance.

PCA remains valuable for understanding and visualizing
behavioural structure, but is not automatically the preferred
clustering representation.

### Alternative Representation

The family-balanced 20-feature matrix will be retained for
sensitivity testing.

### Excluded Variables

Goals, assists, expected goals, finishing efficiency and
other player-quality or output measures will not determine
the initial archetypes.

Age, club, league, season and player reputation will likewise
not be used as clustering features.

These variables may be examined later to describe or validate
the resulting groups.

### Interpretation

An archetype, if identified, will describe a recurring
combination of attacking behaviours.

It must not be interpreted automatically as a player's
fixed tactical position, quality level or optimal team role.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import pairwise_distances

DATA_DIR = Path("/content")

OUTPUT_DIR = DATA_DIR
NB06_DIR = DATA_DIR

required_files = {
    "raw": DATA_DIR / "winger_style_features_raw.parquet",
    "scaled": DATA_DIR / "winger_style_features_scaled.parquet",
    "balanced": DATA_DIR / "winger_style_features_family_balanced.parquet",
    "specification": DATA_DIR / "winger_style_feature_specification.csv",
    "neighbours": DATA_DIR / "top10_primary_euclidean.parquet"
}

print("=" * 65)
print("NOTEBOOK 07 — INPUT FILE VALIDATION")
print("=" * 65)

missing_files = []

for name, path in required_files.items():

    if path.is_file():
        print(f"[FOUND]   {path.name}")
    else:
        print(f"[MISSING] {path.name}")
        missing_files.append(path.name)

print("\nAdditional uploaded files:")
print("-" * 65)

additional_files = sorted([
    path for path in DATA_DIR.iterdir()
    if path.is_file()
    and path.name not in {
        p.name for p in required_files.values()
    }
])

if additional_files:
    for path in additional_files:
        print(path.name)
else:
    print("None")

if missing_files:
    raise FileNotFoundError(
        "\nMissing required files:\n"
        + "\n".join(missing_files)
        + "\n\nUpload these files manually to Colab's "
          "/content/ directory, then rerun this cell."
    )

print("\n" + "=" * 65)
print("SETUP COMPLETE — ALL REQUIRED FILES FOUND")
print("=" * 65)
print(f"Working directory: {DATA_DIR}")
print(f"Required files: {len(required_files)}")
print(f"Additional files: {len(additional_files)}")

In [ ]:
raw_df = pd.read_parquet(
    required_files["raw"]
)

scaled_df = pd.read_parquet(
    required_files["scaled"]
)

balanced_df = pd.read_parquet(
    required_files["balanced"]
)

feature_spec = pd.read_csv(
    required_files["specification"]
)

features = feature_spec["feature"].tolist()

identity_cols = [
    "player",
    "player_id",
    "squad",
    "comp",
    "season",
    "age",
    "born",
    "Minutes"
]

assert len(features) == 20
assert len(set(features)) == 20

assert len(raw_df) == 1396
assert len(scaled_df) == 1396
assert len(balanced_df) == 1396

for df in [raw_df, scaled_df, balanced_df]:

    assert set(
        identity_cols + features
    ).issubset(df.columns)

    assert not df[
        ["player_id", "squad", "season"]
    ].duplicated().any()

    assert np.isfinite(
        df[features].to_numpy(dtype=float)
    ).all()

# Confirm identical observation ordering
for df in [raw_df, balanced_df]:

    pd.testing.assert_frame_equal(
        scaled_df[identity_cols].reset_index(drop=True),
        df[identity_cols].reset_index(drop=True)
    )

X = scaled_df[features].copy()
X_balanced = balanced_df[features].copy()

# Confirm global standardization
assert np.allclose(
    X.mean(axis=0).to_numpy(),
    0,
    atol=1e-6
)

assert np.allclose(
    X.std(axis=0, ddof=0).to_numpy(),
    1,
    atol=1e-6
)

# Confirm family-balanced feature weights
family_counts = feature_spec.groupby(
    "macro_family"
)["feature"].transform("count")

expected_weights = 1 / np.sqrt(
    family_counts.to_numpy()
)

np.testing.assert_allclose(
    feature_spec["family_weight"].to_numpy(),
    expected_weights,
    atol=1e-6
)

expected_balanced = X.mul(
    feature_spec["family_weight"].to_numpy(),
    axis=1
)

np.testing.assert_allclose(
    X_balanced.to_numpy(),
    expected_balanced.to_numpy(),
    rtol=1e-6,
    atol=1e-6
)

print("Modelling matrices validated.")
print("Observations:", len(X))
print("Distinct players:", scaled_df["player_id"].nunique())
print("Features:", len(features))

print("\nBehavioural families:")
display(
    feature_spec.groupby("macro_family")
    .agg(n_features=("feature", "count"))
    .sort_values("macro_family")
)

In [ ]:
pca_artifacts = sorted([
    path
    for path in OUTPUT_DIR.rglob("*")
    if (
        path.is_file()
        and path.suffix.lower() in [
            ".parquet", ".csv", ".joblib", ".pkl"
        ]
        and any(
            term in path.name.lower()
            for term in [
                "pca",
                "pc_scores",
                "loading",
                "component"
            ]
        )
    )
])

print("PCA-related artifacts found:")
print("=" * 60)

for path in pca_artifacts:
    print(path.relative_to(OUTPUT_DIR))

if not pca_artifacts:
    print(
        "No saved PCA artifacts were found. "
        "We will need to restore the Notebook 05 exports."
    )

## 2. Initial Structure Diagnostics

### 2.1 Local Neighbourhood Distances

Notebook 06 identified the closest distinct footballers to
every player-club-season observation.

We use these existing results to investigate local proximity
within the standardized behavioural space.

For each target spell, we examine:

- Distance to the nearest distinct footballer.
- Distance to the fifth-nearest distinct footballer.
- Distance to the tenth-nearest distinct footballer.

These provide descriptive information about how quickly
distance increases as we move away from an observation.

A player with a large tenth-neighbour distance occupies
a relatively sparse region of the observed population.

A player with small distances to several different footballers
occupies a relatively dense region.

These neighbourhoods are defined using distinct candidate
footballers, not repeated seasons of the same candidate.

### Interpretation

Local density is useful for understanding the structure of
the player population.

However, variation in density does not establish the
existence of discrete clusters.

A continuous distribution may contain dense central regions
and sparse tails without dividing naturally into archetypes.

The present analysis is therefore descriptive, not a
formal test of cluster existence.

In [ ]:
neighbours = pd.read_parquet(
    required_files["neighbours"]
)

required_columns = [
    "target_row_idx",
    "candidate_player_id",
    "target_player_id",
    "rank",
    "distance"
]

assert set(required_columns).issubset(
    neighbours.columns
)

assert len(neighbours) == len(X) * 10

assert (
    neighbours["target_player_id"]
    != neighbours["candidate_player_id"]
).all()

assert (
    neighbours.groupby("target_row_idx")
    ["candidate_player_id"].nunique()
    == 10
).all()

# One row per target, one distance per neighbour rank
neighbour_distances = neighbours.pivot(
    index="target_row_idx",
    columns="rank",
    values="distance"
).reindex(range(len(X)))

assert neighbour_distances.shape == (1396, 10)
assert not neighbour_distances.isna().any().any()

local_structure = scaled_df[
    ["player", "player_id", "squad", "season"]
].reset_index(drop=True).copy()

local_structure["d1"] = neighbour_distances[1].to_numpy()
local_structure["d5"] = neighbour_distances[5].to_numpy()
local_structure["d10"] = neighbour_distances[10].to_numpy()

local_structure["d10_minus_d1"] = (
    local_structure["d10"] - local_structure["d1"]
)

local_structure["d10_over_d1"] = (
    local_structure["d10"] / local_structure["d1"]
)

assert (
    local_structure["d1"]
    <= local_structure["d5"]
).all()

assert (
    local_structure["d5"]
    <= local_structure["d10"]
).all()

print("Local neighbourhood distance summary:")

display(
    local_structure[
        ["d1", "d5", "d10", "d10_minus_d1", "d10_over_d1"]
    ].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    ).round(3)
)

print("\nMost isolated observations by tenth-neighbour distance:")

display(
    local_structure.nlargest(10, "d10")[
        [
            "player", "squad", "season",
            "d1", "d5", "d10"
        ]
    ].round(3)
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(
    local_structure["d1"],
    local_structure["d10"],
    alpha=0.35,
    s=20
)

limits = [
    min(local_structure["d1"].min(),
        local_structure["d10"].min()),
    max(local_structure["d1"].max(),
        local_structure["d10"].max())
]

ax.plot(
    limits,
    limits,
    linestyle="--",
    color="gray",
    linewidth=1,
    label="d10 = d1"
)

ax.set_xlabel(
    "Distance to Nearest Distinct Footballer"
)

ax.set_ylabel(
    "Distance to 10th-Nearest Distinct Footballer"
)

ax.set_title(
    "Local Neighbourhood Structure of Winger Playstyles"
)

ax.grid(alpha=0.2)
ax.legend()

plt.tight_layout()
plt.show()


## 2.2 Findings — Local Neighbourhood Structure

Nearest-neighbour distances were examined across all 1,396
player-club-season observations using the primary Euclidean
similarity representation from Notebook 06.

### Population-Level Findings

The median distances are:

- First-nearest distinct footballer: 1.982
- Fifth-nearest distinct footballer: 2.430
- Tenth-nearest distinct footballer: 2.657

For most observations, distance increases relatively gradually
across the first ten distinct candidate footballers.

This suggests that much of the historical winger population
has several reasonably nearby behavioural alternatives.

However, the distribution also contains a substantial
upper tail of locally isolated observations.

### Extreme Player Profiles

Jérémy Doku's Manchester City 2023–24 observation has:

- d1 = 7.147
- d5 = 8.716
- d10 = 9.426

His profile is unusually distant even from its closest
available footballer.

Neymar's PSG 2019–20 observation has:

- d1 = 2.801
- d5 = 6.091
- d10 = 6.923

This represents a different neighbourhood pattern: one
relatively closer alternative followed by considerably
more distant candidates.

Other sparsely surrounded observations include several
historical Messi and Neymar seasons, Cristiano Ronaldo
and Rafa Mir.

### Structural Interpretation

The neighbourhood-distance scatter plot shows a concentrated
main population with a continuous extension toward larger
distances.

No obvious separate groups can be identified from this
diagnostic alone.

However, local-neighbourhood distances do not directly
represent the geometry of the full 20-dimensional space.

They are therefore insufficient to establish whether
discrete playstyle groups exist.

### Next Step

Connect local neighbourhood density to Notebook 05's PCA
representation.

This will reveal where relatively dense and sparse winger
profiles lie along the dominant attacking-behaviour dimensions.

In [ ]:
import joblib
import re

pca_model_path = DATA_DIR / "pca_primary.joblib"
pca_scores_path = DATA_DIR / "pca_primary_scores.parquet"

assert pca_model_path.exists()
assert pca_scores_path.exists()

# Load fitted PCA model from Notebook 05
pca_primary = joblib.load(pca_model_path)

assert hasattr(pca_primary, "components_")
assert pca_primary.n_features_in_ == len(features)
assert pca_primary.n_components_ >= 5

# Validate feature names if stored by sklearn
if hasattr(pca_primary, "feature_names_in_"):
    assert list(pca_primary.feature_names_in_) == features

# Reproduce PCA scores from the original fitted model
projected = pca_primary.transform(X)

pc_columns = [
    f"PC{i}" for i in range(1, 6)
]

pca_coordinates = pd.DataFrame(
    projected[:, :5],
    columns=pc_columns
)

# Read previously exported PCA scores
saved_scores = pd.read_parquet(pca_scores_path)

assert len(saved_scores) == len(X)

# Validate any available observation identifiers
for col in ["player_id", "squad", "season"]:

    if col in saved_scores.columns:
        assert (
            saved_scores[col].reset_index(drop=True)
            .equals(scaled_df[col].reset_index(drop=True))
        ), f"PCA score identity mismatch: {col}"

# Identify common PCA score column naming conventions
saved_pc_columns = {}

for col in saved_scores.columns:

    match = re.fullmatch(
        r"(?:PC|component)[_\s-]*0*(\d+)",
        str(col),
        flags=re.IGNORECASE
    )

    if match:
        saved_pc_columns[int(match.group(1))] = col

# Compare reconstructed scores to Notebook 05's exports
if all(i in saved_pc_columns for i in range(1, 6)):

    exported_array = saved_scores[
        [saved_pc_columns[i] for i in range(1, 6)]
    ].to_numpy(dtype=float)

    np.testing.assert_allclose(
        projected[:, :5],
        exported_array,
        rtol=1e-6,
        atol=1e-6
    )

    print("PCA scores match Notebook 05 exports.")

else:
    print(
        "Saved score column names were not recognized. "
        "Using the original fitted PCA model."
    )
    print("Available columns:", saved_scores.columns.tolist())

print("\nExplained variance from saved PCA model:")

variance = pd.DataFrame({
    "component": pc_columns,
    "explained_variance_pct":
        pca_primary.explained_variance_ratio_[:5] * 100
})

display(variance.round(2))

print(
    "\nCumulative variance (5 PCs):",
    round(
        pca_primary.explained_variance_ratio_[:5].sum() * 100,
        2
    ),
    "%"
)

In [ ]:
# Combine PCA coordinates with neighbourhood diagnostics
structure_pca = pd.concat(
    [
        scaled_df[
            ["player", "player_id", "squad", "season"]
        ].reset_index(drop=True),
        pca_coordinates.reset_index(drop=True),
        local_structure[
            ["d1", "d5", "d10"]
        ].reset_index(drop=True)
    ],
    axis=1
)

fig, axes = plt.subplots(
    1, 2, figsize=(15, 6)
)

# Use shared colour scaling
vmin = structure_pca["d10"].quantile(0.05)
vmax = structure_pca["d10"].quantile(0.95)

projections = [
    ("PC1", "PC2"),
    ("PC3", "PC4")
]

for ax, (x_pc, y_pc) in zip(axes, projections):

    scatter = ax.scatter(
        structure_pca[x_pc],
        structure_pca[y_pc],
        c=structure_pca["d10"],
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        s=20,
        alpha=0.65
    )

    ax.axhline(
        0, color="gray", alpha=0.3, linewidth=0.8
    )

    ax.axvline(
        0, color="gray", alpha=0.3, linewidth=0.8
    )

    ax.set_xlabel(x_pc)
    ax.set_ylabel(y_pc)

    ax.set_title(
        f"{x_pc} vs {y_pc} — Local Neighbourhood Sparsity"
    )

    ax.grid(alpha=0.15)

    # Highlight selected extreme observations
    highlights = [
        ("Jeremy Doku", "2023-2024"),
        ("Lionel Messi", "2018-2019"),
        ("Neymar", "2019-2020")
    ]

    for player, season in highlights:

        mask = (
            (structure_pca["player"] == player)
            & (structure_pca["season"] == season)
        )

        if mask.sum() != 1:
            continue

        row = structure_pca.loc[mask].iloc[0]

        ax.scatter(
            row[x_pc],
            row[y_pc],
            marker="x",
            color="red",
            s=85,
            linewidths=1.5,
            zorder=5
        )

        ax.annotate(
            player,
            (row[x_pc], row[y_pc]),
            xytext=(5, 6),
            textcoords="offset points",
            fontsize=8
        )

fig.colorbar(
    scatter,
    ax=axes,
    label="Distance to 10th-Nearest Distinct Footballer",
    shrink=0.85,
    pad=0.02
)

plt.show()

In [ ]:
# Euclidean distance from the mean standardized feature profile
centre_distance = np.linalg.norm(
    X.to_numpy(),
    axis=1
)

structure_pca["centre_distance"] = centre_distance

# Rank correlation: not assuming a linear relationship
spearman_corr = pd.Series(
    centre_distance
).corr(
    structure_pca["d10"],
    method="spearman"
)

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(
    structure_pca["centre_distance"],
    structure_pca["d10"],
    alpha=0.4,
    s=23
)

ax.set_xlabel(
    "Distance from Population Mean — 20D Standardized Space"
)

ax.set_ylabel(
    "Distance to 10th-Nearest Distinct Footballer"
)

ax.set_title(
    "Global Profile Extremity vs Local Neighbourhood Sparsity"
)

ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print(
    "Spearman correlation:",
    round(spearman_corr, 3)
)

print("\nMost distant observations from population mean:")

display(
    structure_pca.nlargest(
        10, "centre_distance"
    )[
        [
            "player",
            "squad",
            "season",
            "centre_distance",
            "d1",
            "d10"
        ]
    ].round(3)
)

## 2.3 Findings — PCA Geometry and Local Isolation

The original PCA model from Notebook 05 was successfully
restored and validated against its exported scores.

The first five principal components explain 85.23% of
the total standardized feature variance.

### Structure in PCA Space

The PC1–PC2 and PC3–PC4 projections show a broadly
continuous population with a dense central region and
increasingly sparse extremes.

No visually obvious separation into multiple large,
discrete groups is apparent.

Highly isolated observations occupy different regions
of the PCA space.

Jérémy Doku's Manchester City 2023–24 spell lies at
an exceptionally positive PC1 value, reflecting its
carry-led attacking penetration.

Neymar and Lionel Messi occupy unusually high PC2
and negative PC3 regions, corresponding to different
combinations of creative progression, passing and
attacking involvement.

These observations suggest that stylistic extremity
can arise from different behavioural dimensions.

### Global Extremity and Local Sparsity

Distance from the mean standardized 20-feature profile
is strongly associated with distance to the tenth-nearest
distinct footballer.

Spearman rank correlation: 0.812.

More globally extreme observations therefore tend to
have sparser local neighbourhoods.

However, both measures are derived from the same
feature geometry. Their relationship is descriptive,
not an independent validation of tactical similarity.

### Implications for Clustering

The current evidence is consistent with continuous
behavioural variation and sparse extremes.

It does not establish the absence of meaningful groups.

A clustering algorithm may still identify useful
behavioural segments within an overlapping population.

However, these segments should not automatically be
interpreted as naturally separated football archetypes.

### Next Step

Evaluate an initial K-Means baseline across a limited
range of cluster counts, using the complete standardized
20-feature representation.

Examine separation, cluster sizes and assignment ambiguity
before selecting any candidate model for football
interpretation.


# 3. Evaluating Candidate Cluster Structure

## 3.1 K-Means Baseline

K-Means partitions observations into K groups by
minimizing the sum of squared Euclidean distances
between observations and their assigned centroids.

The baseline is applied directly to the 20 globally
standardized behavioural features.

No PCA dimensionality reduction is applied before
clustering.

### Candidate Cluster Counts

We initially evaluate K = 2 through K = 10.

This is an exploratory range, not an assumption that
the population contains between two and ten natural
football archetypes.

### Evaluation Metrics

**1. Silhouette coefficient**

Measures how close an observation is to its assigned
cluster relative to alternative clusters.

Values near 1 indicate stronger relative separation;
values near 0 suggest overlapping or ambiguous
assignments; negative values indicate that another
cluster is closer on average.

The mean silhouette coefficient provides a summary,
while the proportion of negative individual silhouette
values helps identify ambiguous assignments.

**2. Calinski–Harabasz index**

Compares between-cluster dispersion with within-cluster
dispersion. Higher values indicate greater relative
separation under this criterion.

**3. Davies–Bouldin index**

Summarizes within-cluster scatter relative to
between-cluster distances. Lower values are preferred
under this criterion.

**4. Inertia**

Measures total within-cluster squared Euclidean distance.

Inertia always decreases or remains unchanged as K
increases, so it cannot independently identify
the appropriate number of clusters.

**5. Cluster-size distribution**

Examines whether a model produces reasonably populated
groups or isolates very small sets of unusual observations.

### Interpretation Rules

No cluster count will be selected using a single
internal validation score.

A partition must also be evaluated for:

- Distinct and interpretable attacking behaviours.
- Within-cluster variation.
- Sensitivity to extreme observations.
- Stability under alternative representations.
- Ambiguous or transitional player profiles.

Internal clustering metrics indicate geometric
properties of a partition, not established football
archetypes.

The first experiment will identify candidate
configurations for deeper investigation.

In [ ]:
from sklearn.cluster import KMeans

from sklearn.metrics import (
    silhouette_samples,
    calinski_harabasz_score,
    davies_bouldin_score
)

# Primary standardized 20-feature representation
X_array = X.to_numpy(dtype=float)

assert X_array.shape == (1396, 20)
assert np.isfinite(X_array).all()

cluster_range = range(2, 11)

kmeans_models = {}
kmeans_results = []

for k in cluster_range:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=30,
        max_iter=500
    )

    labels = model.fit_predict(X_array)

    # Individual silhouette values, not just the average
    individual_silhouettes = silhouette_samples(
        X_array,
        labels,
        metric="euclidean"
    )

    cluster_sizes = np.bincount(
        labels,
        minlength=k
    )

    kmeans_models[k] = model

    kmeans_results.append({
        "k": k,
        "silhouette_mean": individual_silhouettes.mean(),
        "negative_silhouette_pct": (
            (individual_silhouettes < 0).mean() * 100
        ),
        "calinski_harabasz": calinski_harabasz_score(
            X_array, labels
        ),
        "davies_bouldin": davies_bouldin_score(
            X_array, labels
        ),
        "inertia": model.inertia_,
        "smallest_cluster": cluster_sizes.min(),
        "largest_cluster": cluster_sizes.max(),
        "largest_cluster_pct": (
            cluster_sizes.max() / len(X_array) * 100
        ),
        "cluster_sizes": sorted(
            cluster_sizes.tolist(),
            reverse=True
        )
    })

    print(f"Completed K={k}")

kmeans_evaluation = pd.DataFrame(kmeans_results)

display(
    kmeans_evaluation[
        [
            "k",
            "silhouette_mean",
            "negative_silhouette_pct",
            "calinski_harabasz",
            "davies_bouldin",
            "inertia",
            "smallest_cluster",
            "largest_cluster_pct"
        ]
    ].round(3)
)

print("\nCluster-size distributions:")

display(
    kmeans_evaluation[
        ["k", "cluster_sizes"]
    ]
)

In [ ]:
fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 5)
)

# 1. Silhouette score
axes[0].plot(
    kmeans_evaluation["k"],
    kmeans_evaluation["silhouette_mean"],
    marker="o"
)

axes[0].set_title("Mean Silhouette Coefficient")
axes[0].set_xlabel("Number of Clusters (K)")
axes[0].set_ylabel("Mean Silhouette")
axes[0].grid(alpha=0.2)

# 2. Inertia / elbow
axes[1].plot(
    kmeans_evaluation["k"],
    kmeans_evaluation["inertia"],
    marker="o"
)

axes[1].set_title("K-Means Inertia")
axes[1].set_xlabel("Number of Clusters (K)")
axes[1].set_ylabel("Within-Cluster Sum of Squares")
axes[1].grid(alpha=0.2)

# 3. Assignment ambiguity
axes[2].plot(
    kmeans_evaluation["k"],
    kmeans_evaluation["negative_silhouette_pct"],
    marker="o"
)

axes[2].set_title("Negative Silhouette Assignments")
axes[2].set_xlabel("Number of Clusters (K)")
axes[2].set_ylabel("Observations with Negative Silhouette (%)")
axes[2].grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("Highest mean silhouette configurations:")
display(
    kmeans_evaluation.sort_values(
        "silhouette_mean",
        ascending=False
    )[
        [
            "k",
            "silhouette_mean",
            "negative_silhouette_pct",
            "smallest_cluster",
            "largest_cluster_pct"
        ]
    ].head(5).round(3)
)


## 3.2 Findings — Initial K-Means Evaluation

K-Means was evaluated across K = 2 to K = 10 using
the complete 20-feature standardized representation.

### Separation

K = 2 achieves the highest mean silhouette coefficient
(0.199), with approximately 1.0% negative silhouette
assignments.

This provides evidence of a relatively consistent broad
partition, although absolute separation remains modest.

K = 3, K = 4 and K = 5 produce similar mean silhouette
scores of approximately 0.147–0.149.

Higher values of K generally exhibit weaker average
silhouette separation.

### Cluster Sizes

K = 2 divides the sample into groups containing
764 and 632 player spells.

K = 5 produces five reasonably populated groups:

352, 291, 278, 266 and 209 observations.

No extremely small outlier-only group emerges at these
configurations.

### Other Validation Criteria

Inertia decreases continuously with increasing K,
without a compelling unique elbow.

The Calinski–Harabasz index favours K = 2.

The Davies–Bouldin index is marginally lower for K = 5
than K = 2.

These criteria do not establish an unambiguous number
of naturally occurring groups.

### Preliminary Interpretation

The evidence is consistent with a relatively prominent
broad division and weaker finer-scale segmentation.

However, the clustering metrics measure geometric
properties of the feature space, not football meaning.

Balanced groups and positive silhouette scores do not
establish that discrete natural archetypes exist.

### Decision

K = 2 and K = 5 will be carried forward as exploratory
configurations.

K = 2 represents the strongest broad partition.

K = 5 provides a more granular segmentation to investigate.

Neither configuration is selected as the final archetype
model at this stage.

The next analysis examines their behavioural signatures,
within-group consistency and PCA-space geometry.


## 3.3 Behavioural Interpretation of Candidate Partitions

K-Means centroids represent the average standardized
behavioural profile of observations assigned to each group.

Because all 20 features were globally standardized, a
positive centroid value indicates above-population-average
behaviour on that feature, while a negative value indicates
below-population-average behaviour.

These values do not measure player quality.

### Analysis

For K = 2, we inspect the features with the largest
differences between the two centroids.

For K = 5, we inspect the complete centroid profile
across all 20 features and identify the strongest positive
and negative behavioural tendencies of each group.

We also examine silhouette distributions within clusters
to identify groups with ambiguous assignments.

Finally, we project cluster labels onto the saved PCA
coordinates from Notebook 05.

PCA is used only for visualization; cluster assignments
remain based on the full standardized 20-feature space.

### Interpretation Rules

A centroid's strongest features describe the average
behaviour of its assigned observations.

They do not imply that every player in the group exhibits
that behaviour.

Cluster names will not be assigned until we have reviewed
the actual feature signatures and representative footballers.

In [ ]:
cluster_diagnostics = {}

for k in [2, 5]:

    model = kmeans_models[k]
    labels = model.labels_

    silhouettes = silhouette_samples(
        X_array,
        labels,
        metric="euclidean"
    )

    centroid_distances = np.linalg.norm(
        X_array - model.cluster_centers_[labels],
        axis=1
    )

    diagnostics = pd.DataFrame({
        "cluster": labels,
        "player_id": scaled_df["player_id"].to_numpy(),
        "silhouette": silhouettes,
        "centroid_distance": centroid_distances
    })

    cluster_summary = diagnostics.groupby(
        "cluster"
    ).agg(
        n_spells=("player_id", "size"),
        n_distinct_players=("player_id", "nunique"),
        mean_silhouette=("silhouette", "mean"),
        negative_silhouette_pct=(
            "silhouette",
            lambda s: (s < 0).mean() * 100
        ),
        low_silhouette_pct=(
            "silhouette",
            lambda s: (s < 0.10).mean() * 100
        ),
        median_centroid_distance=(
            "centroid_distance", "median"
        )
    )

    cluster_diagnostics[k] = cluster_summary

    print("\n" + "=" * 65)
    print(f"K = {k} — Within-Cluster Diagnostics")
    print("=" * 65)

    display(cluster_summary.round(3))

In [ ]:
k2_model = kmeans_models[2]

k2_centroids = pd.DataFrame(
    k2_model.cluster_centers_,
    columns=features
)

family_map = (
    feature_spec
    .set_index("feature")["macro_family"]
)

k2_feature_contrasts = pd.DataFrame({
    "feature": features,
    "macro_family": family_map.loc[features].to_numpy(),
    "cluster_0_mean_z": k2_centroids.loc[0].to_numpy(),
    "cluster_1_mean_z": k2_centroids.loc[1].to_numpy()
})

k2_feature_contrasts["difference_1_minus_0"] = (
    k2_feature_contrasts["cluster_1_mean_z"]
    - k2_feature_contrasts["cluster_0_mean_z"]
)

k2_feature_contrasts["abs_difference"] = (
    k2_feature_contrasts["difference_1_minus_0"].abs()
)

k2_feature_contrasts = (
    k2_feature_contrasts
    .sort_values("abs_difference", ascending=False)
)

print("Features most strongly separating the K=2 centroids:")

display(
    k2_feature_contrasts.head(15).round(3)
)

# Visualize the 12 largest centroid differences
plot_df = (
    k2_feature_contrasts.head(12)
    .sort_values("difference_1_minus_0")
)

fig, ax = plt.subplots(figsize=(11, 7))

ax.barh(
    plot_df["feature"],
    plot_df["difference_1_minus_0"]
)

ax.axvline(
    0,
    color="black",
    linewidth=0.8
)

ax.set_xlabel(
    "Centroid Difference (Cluster 1 minus Cluster 0), Z Units"
)

ax.set_title(
    "K=2 — Behavioural Features Separating the Two Groups"
)

ax.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
k5_model = kmeans_models[5]

k5_centroids = pd.DataFrame(
    k5_model.cluster_centers_,
    columns=features,
    index=[f"Cluster {i}" for i in range(5)]
)

# Arrange rows by behavioural family
ordered_features = (
    feature_spec
    .sort_values(["macro_family", "feature"])
    ["feature"]
    .tolist()
)

heatmap_data = k5_centroids[ordered_features].T

fig, ax = plt.subplots(figsize=(11, 10))

vlim = max(
    0.5,
    np.abs(heatmap_data.to_numpy()).max()
)

im = ax.imshow(
    heatmap_data.to_numpy(),
    aspect="auto",
    cmap="RdBu_r",
    vmin=-vlim,
    vmax=vlim
)

ax.set_xticks(np.arange(5))
ax.set_xticklabels(k5_centroids.index)

ax.set_yticks(np.arange(len(ordered_features)))
ax.set_yticklabels([
    f"{feature}  [{family_map.loc[feature]}]"
    for feature in ordered_features
])

ax.set_title(
    "K=5 — Standardized Behavioural Centroid Profiles"
)

fig.colorbar(
    im,
    ax=ax,
    label="Cluster Mean Standardized Feature Value",
    shrink=0.8
)

plt.tight_layout()
plt.show()


# Summarize strongest tendencies within each cluster
profile_rows = []

for c in range(5):

    profile = k5_centroids.loc[f"Cluster {c}"]

    strongest_high = profile.nlargest(4)
    strongest_low = profile.nsmallest(4)

    profile_rows.append({
        "cluster": c,
        "n_spells": int(
            (k5_model.labels_ == c).sum()
        ),
        "strongest_above_average": "; ".join(
            f"{f} ({v:+.2f})"
            for f, v in strongest_high.items()
        ),
        "strongest_below_average": "; ".join(
            f"{f} ({v:+.2f})"
            for f, v in strongest_low.items()
        )
    })

k5_profile_summary = pd.DataFrame(profile_rows)

display(k5_profile_summary)

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 6)
)

for ax, k in zip(axes, [2, 5]):

    model = kmeans_models[k]
    labels = model.labels_

    for c in range(k):

        mask = labels == c

        ax.scatter(
            pca_coordinates.loc[mask, "PC1"],
            pca_coordinates.loc[mask, "PC2"],
            s=16,
            alpha=0.45,
            color=plt.get_cmap("tab10")(c),
            label=f"Cluster {c}"
        )

    # Project the FULL-SPACE centroids using Notebook 05 PCA.
    centroid_df = pd.DataFrame(
        model.cluster_centers_,
        columns=features
    )

    projected_centroids = pca_primary.transform(
        centroid_df
    )

    ax.scatter(
        projected_centroids[:, 0],
        projected_centroids[:, 1],
        marker="X",
        s=170,
        color="black",
        edgecolor="white",
        linewidth=1,
        zorder=5
    )

    for c, coords in enumerate(projected_centroids):

        ax.annotate(
            str(c),
            (coords[0], coords[1]),
            xytext=(7, 7),
            textcoords="offset points",
            fontsize=11,
            fontweight="bold"
        )

    ax.set_title(f"K={k} — Clusters Projected onto PCA")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(alpha=0.15)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


## 3.4 Findings — Behavioural Interpretation of K-Means Partitions

The K = 2 solution primarily separates more advanced,
carry-oriented attacking involvement from comparatively
passing-oriented and deeper attacking activity.

This broadly reproduces the dominant PC1 distinction
identified in Notebook 05.

The K = 5 solution provides more granular behavioural
profiles.

Preliminary descriptions are:

- Cluster 0: Penalty-area-oriented attackers and shooters.
- Cluster 1: Crossing-oriented wide players.
- Cluster 2: Deeper passing-oriented progressors.
- Cluster 3: Direct carrying and dribbling attackers.
- Cluster 4: Advanced creative passers.

These descriptions are provisional and based on centroid
behaviour, not independently validated football archetypes.

### Within-Cluster Separation

The five clusters exhibit unequal silhouette performance.

Cluster 0 has the highest mean silhouette (0.202).
Cluster 2 follows with a mean of 0.181.

Clusters 1 and 3 have weaker mean silhouettes of
0.127 and 0.123.

Cluster 4 has the lowest mean silhouette (0.082).

Approximately 60.3% of Cluster 4 observations have
silhouette values below 0.10, while 17.7% have
negative silhouettes.

These values indicate that many Cluster 4 assignments
are poorly separated from alternative groups.

### PCA Interpretation

The K = 5 partition subdivides the continuous
PC1-PC2 playstyle space into several overlapping
behavioural regions.

The results suggest potentially useful descriptive
segmentation, but do not demonstrate five discrete
natural populations.

### Next Step

Evaluate the football interpretability and assignment
ambiguity of individual player spells.

Cluster descriptions will remain provisional until
representative observations, boundary cases and
robustness have been examined.

In [ ]:
k5_labels = kmeans_models[5].labels_

cluster_contrasts = {}

for c in range(5):

    cluster_mean = X.loc[
        k5_labels == c
    ].mean()

    other_mean = X.loc[
        k5_labels != c
    ].mean()

    contrast = pd.DataFrame({
        "feature": features,
        "cluster_mean_z": cluster_mean.loc[features].to_numpy(),
        "other_clusters_mean_z": other_mean.loc[features].to_numpy()
    })

    contrast["difference"] = (
        contrast["cluster_mean_z"]
        - contrast["other_clusters_mean_z"]
    )

    cluster_contrasts[c] = contrast

    print("\n" + "=" * 65)
    print(f"CLUSTER {c} — DISTINCTIVE BEHAVIOURS")
    print("=" * 65)

    print("\nMost elevated relative to other clusters:")

    display(
        contrast.nlargest(
            5, "difference"
        ).round(3)
    )

    print("\nMost reduced relative to other clusters:")

    display(
        contrast.nsmallest(
            5, "difference"
        ).round(3)
    )

In [ ]:
from sklearn.metrics import silhouette_samples

k5_model = kmeans_models[5]
k5_labels = k5_model.labels_

k5_silhouettes = silhouette_samples(
    X_array,
    k5_labels,
    metric="euclidean"
)

# Euclidean distances from every observation to all centroids
centroid_distance_matrix = pairwise_distances(
    X_array,
    k5_model.cluster_centers_,
    metric="euclidean"
)

sorted_centroid_distances = np.sort(
    centroid_distance_matrix,
    axis=1
)

assigned_distance = sorted_centroid_distances[:, 0]
second_distance = sorted_centroid_distances[:, 1]

# Normalized gap to the next-closest centroid
centroid_margin = (
    (second_distance - assigned_distance)
    / second_distance
)

assert np.array_equal(
    centroid_distance_matrix.argmin(axis=1),
    k5_labels
)

cluster_members = scaled_df[
    [
        "player",
        "player_id",
        "squad",
        "season",
        "Minutes"
    ]
].reset_index(drop=True).copy()

cluster_members["cluster"] = k5_labels
cluster_members["silhouette"] = k5_silhouettes
cluster_members["centroid_distance"] = assigned_distance
cluster_members["centroid_margin"] = centroid_margin

print("Cluster assignment certainty diagnostics:")

display(
    cluster_members.groupby("cluster").agg(
        n_spells=("player_id", "size"),
        median_centroid_margin=("centroid_margin", "median"),
        pct_margin_below_005=(
            "centroid_margin",
            lambda s: (s < 0.05).mean() * 100
        ),
        mean_silhouette=("silhouette", "mean")
    ).round(3)
)

for c in range(5):

    members = cluster_members[
        cluster_members["cluster"] == c
    ]

    print("\n" + "=" * 65)
    print(f"CLUSTER {c} — REPRESENTATIVE FOOTBALLERS")
    print("=" * 65)

    print("\nClosest to centroid (distinct footballers):")

    central = (
        members.sort_values("centroid_distance")
        .drop_duplicates("player_id")
        .head(5)
    )

    display(
        central[
            [
                "player", "squad", "season",
                "centroid_distance", "silhouette"
            ]
        ].round(3)
    )

    print("\nHighest silhouette (distinct footballers):")

    separated = (
        members.sort_values(
            "silhouette",
            ascending=False
        )
        .drop_duplicates("player_id")
        .head(3)
    )

    display(
        separated[
            [
                "player", "squad", "season",
                "silhouette", "centroid_margin"
            ]
        ].round(3)
    )

In [ ]:
players_to_inspect = [
    ("Jeremy Doku", "Manchester City", "2023-2024"),
    ("Vinicius Júnior", "Real Madrid", "2022-2023"),
    ("Lionel Messi", "Barcelona", "2018-2019"),
    ("Neymar", "Paris S-G", "2019-2020"),
    ("Mohamed Salah", "Liverpool", "2017-2018"),
    ("Kaoru Mitoma", "Brighton", "2023-2024"),
    ("Gonçalo Guedes", "Valencia", "2020-2021"),
    ("Mathieu Dossevi", "Toulouse", "2019-2020"),
    ("Ismaila Sarr", "Rennes", "2018-2019")
]

selected_rows = []

for player, squad, season in players_to_inspect:

    mask = (
        (cluster_members["player"] == player)
        & (cluster_members["squad"] == squad)
        & (cluster_members["season"] == season)
    )

    matches = cluster_members.loc[mask]

    if len(matches) != 1:
        print(
            f"Skipping ambiguous/missing case: "
            f"{player}, {squad}, {season}"
        )
        continue

    selected_rows.append(matches.iloc[0])

if selected_rows:
    familiar_players_df = pd.DataFrame(selected_rows)

    display(
        familiar_players_df[
            [
                "player", "season", "cluster",
                "silhouette",
                "centroid_distance",
                "centroid_margin"
            ]
        ].round(3)
    )

# Separate descriptive check:
# How often does a footballer occupy different clusters
# across his available historical spells?

player_cluster_history = (
    cluster_members.groupby("player_id")
    .agg(
        n_spells=("season", "size"),
        n_distinct_clusters=("cluster", "nunique")
    )
)

multi_spell = player_cluster_history[
    player_cluster_history["n_spells"] >= 2
]

print("\nWithin-player historical cluster variation:")
print("Players with >= 2 qualifying spells:", len(multi_spell))
print(
    "Players occupying >= 2 clusters:",
    int((multi_spell["n_distinct_clusters"] >= 2).sum())
)

print(
    "Proportion crossing clusters:",
    round(
        (multi_spell["n_distinct_clusters"] >= 2).mean() * 100,
        2
    ),
    "%"
)

display(
    multi_spell["n_distinct_clusters"]
    .value_counts()
    .sort_index()
    .rename_axis("n_distinct_clusters")
    .reset_index(name="n_players")
)


## 3.5 Findings — Player-Level Interpretability and Assignment Ambiguity

The K = 5 partition was examined through cluster-versus-rest
feature contrasts, centroid-representative player spells,
silhouette scores, centroid margins and historical
within-player variation.

### 3.5.1 Behavioural Identities

The five clusters display interpretable average attacking
behaviours.

**Cluster 0 — Penalty-Area Attacking and Shooting**

The strongest positive contrasts are:

- Box-penetration touch share: +1.041
- Shooting propensity: +0.763
- Carry box-entry share: +0.734

Crossing volume and progressive passing are comparatively low.

Representative spells include Mikel Oyarzabal (2018–19)
and Richarlison (2018–19).

This group emphasizes penalty-area attacking involvement
rather than extensive creative distribution.

**Cluster 1 — Crossing-Oriented Wide Play**

The strongest positive contrasts are:

- Cross-pass share: +1.561
- Crosses per 90: +1.411
- Long-pass share: +1.257

Penalty-area penetration is comparatively low.

Representative spells include Álex Berenguer (2021–22)
and Darko Lazović (2022–23).

This group exhibits a relatively delivery-oriented profile.

**Cluster 2 — Deeper Passing Progression**

The strongest positive contrasts include progressive passes
per 90 (+0.784) and progressive pass share (+0.634).

The strongest negative contrasts include:

- Carry progression share: -1.603
- Attacking-third touch share: -1.537
- Progressive reception share: -1.509

Representative spells include Carlos Soler (2018–19)
and Brais Méndez (2019–20).

This group combines comparatively greater progressive
passing with reduced advanced attacking and carrying activity.

**Cluster 3 — Direct Carrying and Dribbling**

The strongest positive contrasts are:

- Carries into the box per 90: +1.705
- Progressive carry share: +1.468
- Progressive carries per 90: +1.367
- Take-on propensity: +1.338

Representative spells include Takefusa Kubo (2022–23)
and Lucas Ocampos (2019–20).

This group emphasizes direct ball progression through
carrying and dribbling.

**Cluster 4 — Advanced Creative Progression**

The strongest positive contrasts are:

- Passes into the box per 90: +1.547
- Progressive passes per 90: +1.459
- Progressive pass share: +1.155
- Progressive carries per 90: +0.858

Representative spells include Willian (2017–18)
and Philippe Coutinho (2017–18).

This group combines relatively high creative passing
activity with additional progressive attacking involvement.

All cluster descriptions remain provisional.

The feature contrasts describe differences within the
same data used to construct the clusters. They therefore
support interpretation rather than independent validation.

### 3.5.2 Unequal Cluster Separation

The five groups differ substantially in their assignment
quality.

| Cluster | Mean silhouette | Negative silhouette (%) | Median centroid margin |
|---|---:|---:|---:|
| 0 | 0.202 | 0.00 | 0.274 |
| 1 | 0.127 | 6.19 | 0.205 |
| 2 | 0.181 | 1.80 | 0.280 |
| 3 | 0.123 | 12.78 | 0.269 |
| 4 | 0.082 | 17.70 | 0.181 |

Clusters 0 and 2 exhibit comparatively stronger separation.

Cluster 4 is the least well-separated group.

Approximately 60.3% of Cluster 4 observations have
silhouette values below 0.10.

This suggests considerable behavioural overlap with
other clusters.

However, cluster-level averages do not describe the
certainty of every individual observation.

### 3.5.3 Representative Players Versus Extreme Members

Individual player spells can occupy the same cluster
while differing substantially in their proximity to
its centroid.

Within Cluster 3:

- Takefusa Kubo (2022–23): centroid distance 1.278
- Vinícius Júnior (2022–23): centroid distance 6.098
- Jérémy Doku (2023–24): centroid distance 11.361
- Mohamed Salah (2017–18): centroid distance 4.297

Doku is classified within the direct-carrying group
but remains exceptionally distant from its average profile.

Salah also belongs to Cluster 3, although his silhouette
of 0.056 indicates weak separation from alternative groups.

This demonstrates that cluster membership alone does not
adequately describe a footballer's attacking profile.

Future player-level interpretations should distinguish:

1. Assigned cluster.
2. Distance to the assigned centroid.
3. Distance to the nearest alternative centroid.
4. Silhouette and assignment ambiguity.
5. Feature-level differences from the assigned and
   alternative centroids.

Centroid margins and silhouette values are diagnostics,
not calibrated membership probabilities.

### 3.5.4 Historical Within-Player Variation

Of 329 footballers with at least two qualifying spells:

- 154 remain within a single cluster.
- 142 occupy two different clusters.
- 31 occupy three different clusters.
- 2 occupy four different clusters.

Overall, 175 of 329 footballers (53.19%) occupy more than
one cluster during their observed historical spells.

This supports treating player-club-season as the
observational unit rather than assuming that every player
possesses one permanent archetype.

However, cross-cluster movement cannot automatically be
attributed to genuine stylistic evolution.

Alternative explanations include tactical context,
seasonal variation, data limitations and ambiguous
cluster boundaries.

The number of observed spells also affects the
opportunity for a player to occupy multiple clusters.

### 3.5.5 Interim Conclusion

The K = 5 solution produces five interpretable average
behavioural profiles.

However, the groups are not equally well separated,
and some individual memberships are ambiguous.

The findings support further investigation of K = 5 as
a descriptive segmentation.

They do not yet establish five naturally distinct
football archetypes.

K = 2 remains the stronger broad geometric partition,
while the continuous PCA representation remains
an important alternative.

No final clustering configuration is selected at
this stage.

Before final model selection, the candidate segmentation
must undergo robustness testing, including sensitivity
to feature-family weighting and repeated-player sampling.

In [ ]:
from pathlib import Path

EXPORT_DIR = Path("/content")

# Individual observations, assignments and diagnostics
cluster_members.to_csv(
    EXPORT_DIR / "k5_exploratory_members.csv",
    index=False
)

# Standardized centroid profiles
k5_centroids.to_csv(
    EXPORT_DIR / "k5_exploratory_centroids.csv"
)

# Cluster-versus-rest feature contrasts
contrast_export = pd.concat(
    [
        df.assign(cluster=cluster)
        for cluster, df in cluster_contrasts.items()
    ],
    ignore_index=True
)

contrast_export.to_csv(
    EXPORT_DIR / "k5_exploratory_feature_contrasts.csv",
    index=False
)

print("Exploratory clustering artifacts saved.")

for filename in [
    "k5_exploratory_members.csv",
    "k5_exploratory_centroids.csv",
    "k5_exploratory_feature_contrasts.csv"
]:
    path = EXPORT_DIR / filename
    print(f"{filename}: {path.exists()}")


## 3.6 Robustness to Feature-Family Weighting

The initial K-Means investigation has identified
interpretable behavioural profiles, but their stability
under alternative feature weighting remains untested.

The primary representation assigns equal weight to
each standardized feature.

Because the 20 features are distributed across eight
behavioural families with unequal feature counts,
some families contribute more dimensions to the
Euclidean distance calculation.

Notebook 04 also produced a family-balanced
representation, in which each standardized feature
is multiplied by the inverse square root of the
number of features in its family.

This reduces the direct effect of unequal family sizes
on squared Euclidean distances.

### Objective

Refit K-Means under the family-balanced representation
and compare the resulting observation assignments
against the original standardized representation.

Candidate values K = 2, 3, 4, 5 and 6 are retained
for this sensitivity analysis.

### Evaluation

- Adjusted Rand Index (ARI) measures agreement
  between partitions after adjusting for chance.
- Normalized Mutual Information (NMI) measures
  shared information between cluster assignments.
- Hungarian label matching provides an interpretable
  matched-assignment percentage.
- Cross-tabulations identify which original groups
  are split or reorganized.

Cluster identifiers are arbitrary and must be aligned
before directly comparing labels.

Internal silhouette scores will also be calculated
within the family-balanced feature space.

Silhouette scores from differently weighted spaces
should not be interpreted as directly comparable
measurements of identical geometry.

### Interpretation

This analysis measures sensitivity to feature-family
weighting.

Agreement would support robustness to that particular
modelling decision.

It would not independently prove that discrete natural
football archetypes exist.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score
)
from scipy.optimize import linear_sum_assignment

# Use the existing Notebook 04 family-balanced features
X_family = X_balanced[features].to_numpy(dtype=float)

assert X_family.shape == X_array.shape
assert np.isfinite(X_family).all()

family_kmeans_models = {}
robustness_results = []

for k in [2, 3, 4, 5, 6]:

    primary_labels = kmeans_models[k].labels_

    family_model = KMeans(
        n_clusters=k,
        n_init=30,
        random_state=42,
        max_iter=500
    )

    family_labels = family_model.fit_predict(X_family)
    family_kmeans_models[k] = family_model

    # Label-invariant partition comparisons
    ari = adjusted_rand_score(
        primary_labels, family_labels
    )

    nmi = normalized_mutual_info_score(
        primary_labels, family_labels
    )

    # Match arbitrary cluster labels using Hungarian matching
    contingency = pd.crosstab(
        pd.Series(primary_labels),
        pd.Series(family_labels)
    ).reindex(
        index=range(k),
        columns=range(k),
        fill_value=0
    )

    row_idx, col_idx = linear_sum_assignment(
        -contingency.to_numpy()
    )

    matched_agreement = (
        contingency.to_numpy()[row_idx, col_idx].sum()
        / len(primary_labels)
    )

    family_silhouette = silhouette_score(
        X_family, family_labels
    )

    robustness_results.append({
        "k": k,
        "adjusted_rand_index": ari,
        "normalized_mutual_information": nmi,
        "matched_agreement_pct": matched_agreement * 100,
        "family_balanced_silhouette": family_silhouette
    })

robustness_df = pd.DataFrame(robustness_results)

print("Family-weighting robustness results:")
display(robustness_df.round(3))

In [ ]:
primary_labels = kmeans_models[5].labels_
family_labels = family_kmeans_models[5].labels_

contingency = pd.crosstab(
    pd.Series(primary_labels, name="Primary"),
    pd.Series(family_labels, name="Family balanced")
).reindex(
    index=range(5),
    columns=range(5),
    fill_value=0
)

# Align family-balanced IDs to primary cluster IDs
rows, cols = linear_sum_assignment(
    -contingency.to_numpy()
)

family_to_primary = {
    int(family_id): int(primary_id)
    for primary_id, family_id in zip(rows, cols)
}

aligned_family_labels = np.array([
    family_to_primary[label]
    for label in family_labels
])

transition_counts = pd.crosstab(
    pd.Series(primary_labels, name="Original cluster"),
    pd.Series(aligned_family_labels, name="Balanced cluster")
)

transition_pct = pd.crosstab(
    pd.Series(primary_labels, name="Original cluster"),
    pd.Series(aligned_family_labels, name="Balanced cluster"),
    normalize="index"
) * 100

print("K=5 — Cluster transitions (counts):")
display(transition_counts)

print("\nK=5 — Cluster transitions (% of original group):")
display(transition_pct.round(1))

print(
    "\nOverall matched assignment agreement:",
    f"{(primary_labels == aligned_family_labels).mean() * 100:.2f}%"
)


## 3.7 Robustness to Repeated Players and Extreme Observations

The family-weighting experiment indicates substantial
stability of the exploratory K-Means partitions.

At K = 5, the adjusted Rand index is 0.831 and
92.41% of assignments are retained after label alignment.

Clusters 2 and 3 are particularly stable, retaining
97.5% and 98.1% of their original observations.

Cluster 1 is comparatively sensitive, with several
observations moving into Cluster 4.

### Remaining Methodological Concerns

**Repeated footballers**

The 1,396 observations represent 665 distinct footballers.

Footballers with more qualifying historical spells
contribute more observations to the original K-Means
objective.

We will therefore refit K-Means while weighting each
footballer's complete history to contribute equal total
weight.

This preserves historical observations while reducing
the influence of unequal career coverage.

**Extreme attacking profiles**

The population contains several globally unusual
observations, particularly Doku's 2023–24 season.

K-Means minimizes squared Euclidean distances, making
it potentially sensitive to extreme observations.

We will remove the most distant 1% of observations from
model fitting, refit K-Means, and classify all original
observations using the resulting model.

The 1% cutoff is an exploratory sensitivity setting,
not a definition of invalid observations.

### Evaluation

We compare the original K-Means partitions with the
alternative fits using:

- Adjusted Rand Index.
- Hungarian-matched assignment agreement.
- Cluster-level assignment retention.

K = 2 and K = 5 are tested.

The objective is to establish whether the candidate
playstyle groups depend excessively on these modelling
choices, not to maximize internal validation scores.

These experiments use the previously standardized
feature space, isolating the effects of observation
weighting and extreme-profile removal.

In [ ]:
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score

def compare_cluster_assignments(
    original_labels,
    alternative_labels,
    k
):
    """
    Compare partitions after aligning arbitrary cluster IDs.
    """

    original_labels = np.asarray(original_labels)
    alternative_labels = np.asarray(alternative_labels)

    assert len(original_labels) == len(alternative_labels)

    contingency = np.zeros((k, k), dtype=int)

    np.add.at(
        contingency,
        (original_labels, alternative_labels),
        1
    )

    rows, cols = linear_sum_assignment(-contingency)

    # Map alternative labels to original labels
    mapping = {
        int(alternative): int(original)
        for original, alternative in zip(rows, cols)
    }

    aligned = np.array([
        mapping[label]
        for label in alternative_labels
    ])

    ari = adjusted_rand_score(
        original_labels,
        alternative_labels
    )

    agreement = (
        (original_labels == aligned).mean() * 100
    )

    retention = {}

    for c in range(k):
        mask = original_labels == c

        retention[c] = (
            (aligned[mask] == c).mean() * 100
        )

    return {
        "ari": ari,
        "matched_agreement_pct": agreement,
        "retention_by_cluster": retention,
        "aligned_labels": aligned
    }

In [ ]:
# Number of observations contributed by each footballer
spell_counts = (
    scaled_df.groupby("player_id")["player_id"]
    .transform("size")
    .to_numpy()
)

player_weights = 1.0 / spell_counts

assert np.isclose(
    player_weights.sum(),
    scaled_df["player_id"].nunique()
)

player_weighted_models = {}
player_weighting_results = []

for k in [2, 5]:

    weighted_model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=30,
        max_iter=500
    )

    weighted_model.fit(
        X_array,
        sample_weight=player_weights
    )

    player_weighted_models[k] = weighted_model

    comparison = compare_cluster_assignments(
        kmeans_models[k].labels_,
        weighted_model.labels_,
        k
    )

    player_weighting_results.append({
        "k": k,
        "ARI": comparison["ari"],
        "matched_agreement_pct":
            comparison["matched_agreement_pct"]
    })

    print(f"\nK={k} — Retention by original cluster")

    display(
        pd.DataFrame({
            "cluster": list(
                comparison["retention_by_cluster"].keys()
            ),
            "retained_pct": list(
                comparison["retention_by_cluster"].values()
            )
        }).round(2)
    )

print("\nOverall player-weighting robustness:")

display(
    pd.DataFrame(
        player_weighting_results
    ).round(3)
)

In [ ]:
# Global distance from the standardized population mean
global_distances = np.linalg.norm(
    X_array,
    axis=1
)

cutoff = np.quantile(
    global_distances,
    0.99
)

keep_mask = global_distances <= cutoff

print("Observations retained:", keep_mask.sum())
print("Observations excluded:", (~keep_mask).sum())

print("\nExcluded player spells:")

excluded = (
    scaled_df.loc[
        ~keep_mask,
        ["player", "squad", "season"]
    ]
    .copy()
)

excluded["global_distance"] = global_distances[
    ~keep_mask
]

display(
    excluded.sort_values(
        "global_distance",
        ascending=False
    ).round(3)
)

trimmed_models = {}
trimming_results = []

for k in [2, 5]:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=30,
        max_iter=500
    )

    # Fit without the extreme observations
    model.fit(
        X_array[keep_mask]
    )

    trimmed_models[k] = model

    # Classify the original complete population
    predicted_all = model.predict(X_array)

    comparison = compare_cluster_assignments(
        kmeans_models[k].labels_,
        predicted_all,
        k
    )

    # Also compare agreement only on observations
    # actually retained in the alternative fit
    retained_agreement = (
        kmeans_models[k].labels_[keep_mask]
        == comparison["aligned_labels"][keep_mask]
    ).mean() * 100

    trimming_results.append({
        "k": k,
        "ARI_all_observations": comparison["ari"],
        "matched_agreement_pct":
            comparison["matched_agreement_pct"],
        "retained_observations_agreement_pct":
            retained_agreement
    })

    print(f"\nK={k} — Retention by original cluster")

    display(
        pd.DataFrame({
            "cluster": list(
                comparison["retention_by_cluster"].keys()
            ),
            "retained_pct": list(
                comparison["retention_by_cluster"].values()
            )
        }).round(2)
    )

print("\nExtreme-profile sensitivity results:")

display(
    pd.DataFrame(
        trimming_results
    ).round(3)
)

## 3.8 Final Robustness Findings and Modelling Decision

### Feature-Family Weighting

At K = 5, the family-balanced alternative produces:

- Adjusted Rand Index: 0.831
- Matched assignment agreement: 92.41%

The five-group partition is substantially preserved when
the relative contributions of behavioural feature families
are changed.

The crossing-oriented group is comparatively sensitive,
particularly at its boundary with advanced creative
progression.

### Equal Total Weight per Footballer

The alternative fit assigns equal cumulative sample weight
to each distinct footballer, regardless of how many
qualifying historical spells they contribute.

At K = 5:

- Adjusted Rand Index: 0.709
- Matched assignment agreement: 87.32%
- Unchanged assignments: 1,219 of 1,396

Clusters 3 and 4 retain 100.00% and 99.04% of their
original observations, respectively.

Clusters 0, 1 and 2 retain 81.82%, 83.85% and 76.98%.

These results show that repeated-player representation
meaningfully affects some cluster boundaries.

The weighted fit retains every observation, but does
not remove within-player dependence or re-estimate
the global feature standardization.

### Extreme-Profile Sensitivity

The 14 observations with the greatest standardized
distance from the population mean were excluded
during alternative model fitting.

All original observations were subsequently assigned
using the alternative fitted centroids.

At K = 5:

- Adjusted Rand Index: 0.912
- Matched assignment agreement: 96.06%
- Agreement on the retained fitting population: 96.02%

The broad segmentation is therefore relatively
insensitive to removing the most extreme 1% of
observations during K-Means fitting.

This experiment does not test how the results would
change if feature standardization were also refitted
after removing extreme observations.

### Comparison with K = 2

K = 2 achieves greater geometric separation and
higher agreement across all robustness experiments.

However, its primary behavioural distinction largely
reproduces the dominant attacking-style continuum
already captured by PCA.

K = 5 provides additional interpretable distinctions
between penalty-area attacking, wide delivery,
deeper passing, direct carrying and advanced creativity.

Its additional granularity comes with increased
cluster overlap and some sensitivity to observation
weighting.

### Modelling Decision

K = 5 is retained as the WORKING DESCRIPTIVE TAXONOMY
for downstream winger profiling.

The primary model is the original K-Means fit on
the complete globally standardized 20-feature space.

This decision prioritizes interpretable behavioural
segmentation while retaining the model's documented
limitations.

The results do not establish five naturally separated
football populations.

Accordingly:

1. Cluster labels will describe tendencies rather
   than permanent or exclusive player identities.

2. Continuous PCA dimensions will remain available
   to explain stylistic variation.

3. Full 20-dimensional player similarity will remain
   the primary nearest-neighbour representation.

4. Player profiles will report proximity to their
   assigned centroid and ambiguity relative to
   alternative centroids.

5. The robustness results will accompany the model
   documentation rather than being omitted from
   downstream interpretations.

6. Style membership will not be presented as a
   measurement of player quality or effectiveness.

### Conclusion

Notebook 07 establishes a reproducible and interpretable
five-profile exploratory taxonomy for winger attacking
behaviour, supported by sensitivity analyses and
player-level interpretation.

The taxonomy is suitable for descriptive exploration,
subject to its limitations.

The next stage is to construct individual winger
profiles using the combined similarity, PCA and
clustering representations.

In [ ]:
# ============================================================
# NOTEBOOK 07 — FINAL EXPORTS
# Working descriptive model: Primary K-Means, K=5
# ============================================================

import json
import joblib
import sklearn
from pathlib import Path

from sklearn.metrics import silhouette_samples
from sklearn.metrics import pairwise_distances

EXPORT_DIR = Path("/content")

primary_k5 = kmeans_models[5]
primary_k2 = kmeans_models[2]

n = len(scaled_df)

assert X_array.shape == (1396, 20)
assert n == 1396
assert len(features) == 20
assert scaled_df["player_id"].nunique() == 665

# ------------------------------------------------------------
# 1. Calculate individual assignment diagnostics
# ------------------------------------------------------------

labels_k5 = primary_k5.labels_
labels_k2 = primary_k2.labels_

distances = pairwise_distances(
    X_array,
    primary_k5.cluster_centers_,
    metric="euclidean"
)

assert np.array_equal(
    distances.argmin(axis=1),
    labels_k5
)

sorted_distances = np.sort(distances, axis=1)

assigned_distance = sorted_distances[:, 0]
alternative_distance = sorted_distances[:, 1]

centroid_margin = (
    (alternative_distance - assigned_distance)
    / alternative_distance
)

silhouettes = silhouette_samples(
    X_array,
    labels_k5
)

# ------------------------------------------------------------
# 2. Align alternative model assignments
# ------------------------------------------------------------

alternatives = {
    "family_balanced": family_kmeans_models[5].labels_,
    "equal_player_weight": player_weighted_models[5].labels_,
    "extreme_trimmed": trimmed_models[5].predict(X_array)
}

aligned_alternatives = {}

for name, alternative_labels in alternatives.items():

    comparison = compare_cluster_assignments(
        labels_k5,
        alternative_labels,
        5
    )

    aligned_alternatives[name] = comparison["aligned_labels"]

# ------------------------------------------------------------
# 3. Assemble the player-level assignment dataset
# ------------------------------------------------------------

identity_cols = [
    c for c in [
        "player", "player_id", "squad", "comp",
        "season", "age", "born", "Minutes"
    ]
    if c in scaled_df.columns
]

assignments = (
    scaled_df[identity_cols]
    .reset_index(drop=True)
    .copy()
)

assignments.insert(
    0, "observation_row_idx", np.arange(n)
)

assignments["k2_cluster"] = labels_k2
assignments["k5_cluster"] = labels_k5

assignments["k5_silhouette"] = silhouettes
assignments["k5_centroid_distance"] = assigned_distance
assignments["k5_second_centroid_distance"] = alternative_distance
assignments["k5_centroid_margin"] = centroid_margin

# PCA coordinates from original Notebook 05 model
for pc in ["PC1", "PC2", "PC3", "PC4", "PC5"]:
    assignments[pc] = (
        pca_coordinates[pc]
        .reset_index(drop=True)
        .to_numpy()
    )

for name, aligned_labels in aligned_alternatives.items():
    assignments[f"k5_{name}_aligned"] = aligned_labels

assignments["k5_label_preserved_all_three_tests"] = (
    (assignments["k5_cluster"] == assignments[
        "k5_family_balanced_aligned"
    ])
    & (assignments["k5_cluster"] == assignments[
        "k5_equal_player_weight_aligned"
    ])
    & (assignments["k5_cluster"] == assignments[
        "k5_extreme_trimmed_aligned"
    ])
)

assignments.to_parquet(
    EXPORT_DIR / "winger_k5_profile_assignments.parquet",
    index=False
)

# ------------------------------------------------------------
# 4. Export centroid profiles
# ------------------------------------------------------------

centroids = pd.DataFrame(
    primary_k5.cluster_centers_,
    columns=features
)

centroids.insert(
    0, "cluster", range(5)
)

centroids.to_csv(
    EXPORT_DIR / "winger_k5_centroids_standardized.csv",
    index=False
)

# ------------------------------------------------------------
# 5. Export model robustness summary
# ------------------------------------------------------------

robustness_rows = []

for k in [2, 5]:

    original = kmeans_models[k].labels_

    alternative_models = {
        "family_balanced": family_kmeans_models[k].labels_,
        "equal_player_weight": player_weighted_models[k].labels_,
        "extreme_trimmed": trimmed_models[k].predict(X_array)
    }

    for test_name, alternative in alternative_models.items():

        comparison = compare_cluster_assignments(
            original, alternative, k
        )

        robustness_rows.append({
            "k": k,
            "test": test_name,
            "adjusted_rand_index": comparison["ari"],
            "matched_agreement_pct":
                comparison["matched_agreement_pct"]
        })

robustness_export = pd.DataFrame(robustness_rows)

robustness_export.to_csv(
    EXPORT_DIR / "winger_clustering_robustness.csv",
    index=False
)

# ------------------------------------------------------------
# 6. Save fitted models
# ------------------------------------------------------------

joblib.dump(
    primary_k5,
    EXPORT_DIR / "winger_kmeans_primary_k5.joblib"
)

joblib.dump(
    primary_k2,
    EXPORT_DIR / "winger_kmeans_primary_k2.joblib"
)

joblib.dump(
    family_kmeans_models[5],
    EXPORT_DIR / "winger_kmeans_balanced_k5.joblib"
)

# ------------------------------------------------------------
# 7. Export modelling manifest
# ------------------------------------------------------------

provisional_labels = {
    "0": "Penalty-area attacking and shooting",
    "1": "Crossing-oriented wide play",
    "2": "Deeper passing progression",
    "3": "Direct carrying and dribbling",
    "4": "Advanced creative progression"
}

manifest = {
    "project": "DECODING WINGER PLAYSTYLES",
    "notebook": "07_winger_archetype_discovery",
    "status": "working_descriptive_taxonomy",

    "n_observations": int(n),
    "n_distinct_players": int(
        scaled_df["player_id"].nunique()
    ),

    "feature_count": len(features),
    "feature_order": list(features),

    "primary_representation":
        "20 globally standardized attacking-behaviour features",

    "primary_model":
        "KMeans(n_clusters=5, random_state=42, n_init=30, max_iter=500)",

    "provisional_cluster_names": provisional_labels,

    "limitations": [
        "Clusters are descriptive, not proven natural categories.",
        "Cluster boundaries overlap.",
        "Some assignments are sensitive to footballer weighting.",
        "Multiple observations can belong to the same footballer.",
        "Centroid margins are not membership probabilities.",
        "Clustering features describe style, not player quality."
    ],

    "sklearn_version": sklearn.__version__,

    "outputs": [
        "winger_k5_profile_assignments.parquet",
        "winger_k5_centroids_standardized.csv",
        "winger_clustering_robustness.csv",
        "winger_kmeans_primary_k5.joblib",
        "winger_kmeans_primary_k2.joblib",
        "winger_kmeans_balanced_k5.joblib"
    ]
}

manifest_path = (
    EXPORT_DIR / "winger_clustering_manifest.json"
)

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------
# 8. Verify exports
# ------------------------------------------------------------

print("=" * 65)
print("NOTEBOOK 07 — EXPORT VALIDATION")
print("=" * 65)

for filename in manifest["outputs"] + [
    "winger_clustering_manifest.json"
]:
    path = EXPORT_DIR / filename
    assert path.is_file(), f"Missing export: {filename}"
    print("[SAVED]", filename)

print("\nAssignments preview:")
display(assignments.head())

print("\nRobustness summary:")
display(robustness_export.round(3))

print("\nK=5 cluster counts:")
display(
    assignments["k5_cluster"]
    .value_counts()
    .sort_index()
)